# X18 — two 2026 frontier changes, screened at 30M

**NoPE** (SmolLM3): drop rotary embeddings on every 4th layer. Parameter-neutral by
construction, so a difference cannot be attributed to capacity.

**Gated Residual** (Qwen3.8): data-dependent elementwise gate on the residual path.
Only the gating half is implemented, not the four-branch widening — the
pre-registration says so, and the arm is named `gated` for that reason.

Control vs the two, 500 steps, seed 1337, same data. Pre-registered at
`docs/preregistrations/2026-09-28-X18-frontier-attention.md` (hash 4a4b…).

In [ ]:
import glob, os, shutil, subprocess, sys

work = "/kaggle/working/kalia"
if not os.path.exists(work):
    hits = sorted(glob.glob("/kaggle/input/**/ablate.py", recursive=True))
    assert hits, "attach the kalia-code-dev dataset (must be the NEW version)"
    shutil.copytree(os.path.dirname(hits[0]), work)
os.chdir(work)
print("code from", work)
data = sorted(glob.glob("/kaggle/input/**/train.bin", recursive=True))
assert data, "train.bin not found - attach the kalia-prep-v2b output"
print("data:", data[0])

In [ ]:
print(subprocess.run([sys.executable, "ablate.py",
                       "--arms", "micro-base,micro-nope,micro-gated",
                       "--steps", "500",
                       "--data-dir", str(os.path.dirname(data[0])),
                       "--out-root", "/kaggle/working/out"],
                      capture_output=True, text=True).stdout[-4000:])

In [ ]:
import pandas as pd
for arm in ["micro-base", "micro-nope", "micro-gated"]:
    p = f"/kaggle/working/out/{arm}/val_log.csv"
    try:
        df = pd.read_csv(p)
        print(f"\n{arm}: " + "  ".join(f"{int(r.step)}={r.val_loss:.4f}" for r in df.itertuples()))
    except Exception as exc:
        print(arm, "->", exc)

Read against the pre-registered thresholds:
- **F-1** NoPE val loss ≤ control
- **F-3** gated: val loss ≤ control **and** ≥0.5pp better on a downstream task
- **F-4** the gate opens (mean > 0.05 at the end)

Results are reported as measured. A null at 30M says little about 125B, and the
pre-registration records that caveat in advance.